# Right / left / both-eye comparison on the distance device

Same stimulus (distance 1) shown with the right eye only, the left eye only, and both eyes
(`260928_SAM047_distance-right`, `-left`, `-both`). All three come from one concatenated suite2p
run, so ROI *i* is the same cell in every session. Counts the cells reliable in each condition and
compares amplitude and RF across conditions. Left hemisphere imaged: right eye = contra.

Run this in its own kernel with **only** `radial-analysis` on `sys.path`.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

base_dir = r"C:\Users\acostamendoza\projects\distance-project"
for p in (base_dir, base_dir + r"\radial-analysis"):
    if p not in sys.path:
        sys.path.append(p)

from src.data.RadialDevice import RadialDevice
from src.analysis.CellAnalysis import SessionAnalyzer

%load_ext autoreload
%autoreload 2

## Step 0: load and sanity checks

In [ ]:
mouse, date = "SAM047", "260928"
eyes = ("right", "left", "both")
D = {eye: RadialDevice(mouse_name=mouse, record_date=date, experiment_name=f"distance-{eye}")
     for eye in eyes}

# same concatenated suite2p run -> same ROIs in the same order
for eye in eyes[1:]:
    assert np.array_equal(D[eye].suite2p_id, D["right"].suite2p_id), f"{eye}: ROIs differ from right"
cells = D["right"].suite2p_id
print("cells:", len(cells))

for eye, d in D.items():
    dist, n = np.unique(d.distance_sequence, return_counts=True)
    k = d.get_condition_trials()                      # stimulus trials (control excluded)
    print(f"\n{d.session_name}")
    print(f"  frames={d.spikes.shape[1]}  on_length={d.on_length}  "
          f"angles={np.ravel(d.start_angle)[0]:.1f} -> {np.ravel(d.end_angle)[0]:.1f}")
    print(f"  trials per distance={dict(zip(dist.tolist(), n.tolist()))}  "
          f"monocular_sequence={np.unique(d.monocular_sequence).tolist()}")
    print(f"  on duration={np.mean(d.on_end[k] - d.on_start[k]):.2f}s  "
          f"off_start-on_start={np.mean(d.off_start[k] - d.on_start[k]):.2f}s  "
          f"raise={np.ravel(d.raise_time)[0]}  lower={np.ravel(d.lower_time)[0]}  off={np.ravel(d.off_time)[0]}")

In [ ]:
# common angle axis: shortest on-period across the three sessions
assert len({(np.ravel(d.start_angle)[0], np.ravel(d.end_angle)[0]) for d in D.values()}) == 1, "sweep angles differ"
on_length = min(d.on_length for d in D.values())
for d in D.values():
    d.on_length = on_length
    d.angle_range = np.linspace(d.start_angle, d.end_angle, int(on_length))
angles = np.ravel(D["both"].angle_range)

# pooled normalization: one mu/sd per cell over all three sessions, so amplitudes
# are comparable across R/L/B (RadialDevice z-scores each session on its own).
# Re-run this cell whenever D is rebuilt -- the constructor restores per-session z.
all_spikes = np.concatenate([D[eye].spikes for eye in eyes], axis=1)
mu = all_spikes.mean(axis=1, keepdims=True)
sd = all_spikes.std(axis=1, keepdims=True)
for eye in eyes:
    D[eye].response = gaussian_filter1d((D[eye].spikes - mu) / sd, 1.5, axis=1)

z = (all_spikes - mu) / sd
print("pooled z  mean range:", z.mean(1).min().round(3), z.mean(1).max().round(3),
      "| std range:", z.std(1).min().round(3), z.std(1).max().round(3))

# are raw spikes on the same scale in all three sessions?
for eye in eyes:
    print(f"  {eye:<5} median raw-spike std: {np.median(D[eye].spikes.std(axis=1)):.4g}")
del all_spikes, z

## Step 1: trial tensors

In [ ]:
def trial_tensor(d):
    """cells x trials x angle for the stimulus trials (control excluded); same slicing as get_on_trace"""
    X = []
    for k in d.get_condition_trials():
        on_start, on_end = d.get_on_times(k)
        mask = (d.frames_time > on_start) & (d.frames_time < on_end)
        X.append(d.response[:, mask][:, :d.on_length])
    return np.stack(X, axis=1)

X = {eye: trial_tensor(D[eye]) for eye in eyes}
for eye in eyes:
    print(eye, X[eye].shape)

## Step 2: reliability per condition and categories

In [ ]:
# your standard pipeline per session, cached to the usual pkl. The result is the same
# under per-session or pooled normalization (correlation and argmax are scale-free).
stats = {}
for eye, d in D.items():
    pkl_path = os.path.join(d.data_dir, f"{d.session_name}_dataframe.pkl")
    if os.path.exists(pkl_path):
        stats[eye] = pd.read_pickle(pkl_path)
        print(f"Loaded cached stats: {pkl_path}")
    else:
        sa = SessionAnalyzer(d)
        stats[eye] = sa.analyze_session_stats(n_iterations=200)
        sa.save_dataframe(stats[eye])

In [ ]:
def condition_stats(df):
    # each session holds a single eye condition, so exactly one of the two dicts is filled
    filled = [c for c in ("monocular", "binocular") if df[c].notna().all()]
    assert len(filled) == 1, f"expected one condition per session, got {filled}"
    return pd.DataFrame(df[filled[0]].tolist(), index=df["cell_id"].values)[["reliable", "reliable_corr", "RF"]]

eye_df = pd.concat({eye: condition_stats(stats[eye]) for eye in eyes}, axis=1)
eye_df.columns = [f"{metric}_{eye}" for eye, metric in eye_df.columns]
eye_df.index.name = "cell_id"

CORR_FLOOR = None    # e.g. 0.7 to match the monocular report's mean_corr floor
for eye in eyes:
    resp = eye_df[f"reliable_{eye}"].astype(bool)
    if CORR_FLOOR is not None:
        resp &= eye_df[f"reliable_corr_{eye}"] > CORR_FLOOR
    eye_df[f"resp_{eye}"] = resp

R, L, B = (eye_df[f"resp_{eye}"] for eye in eyes)
eye_df["category"] = np.select([R & L, R & ~L, ~R & L, ~R & ~L & B],
                               ["R&L", "R-only", "L-only", "B-only"], default="none")

n_cells, any_resp = len(eye_df), R | L | B
n_any = any_resp.sum()
print("=" * 50)
print(f" {date}_{mouse} distance: right / left / both")
print(f" Total cells: {n_cells}" + (f" | mean_corr floor: {CORR_FLOOR}" if CORR_FLOOR else ""))
print("-" * 50)
for eye, m in zip(eyes, (R, L, B)):
    print(f" {'Reliable ' + eye:<24}: {m.sum():5d}  ({100 * m.mean():5.1f}% of all cells)")
print(f" {'Reliable in any':<24}: {n_any:5d}  ({100 * n_any / n_cells:5.1f}% of all cells)")
print("-" * 50)
print(f" Among those {n_any} cells:")
for cat, label in [("R-only", "Right only (contra)"), ("L-only", "Left only (ipsi)"),
                   ("R&L", "Both monocular (R & L)"), ("B-only", "Only with both open")]:
    k = (eye_df["category"] == cat).sum()
    print(f" {label:<24}: {k:5d}  ({100 * k / n_any:5.1f}%)")
print("-" * 50)
print(f" R-reliable also reliable under B: {(R & B).sum()} / {R.sum()}")
print(f" L-reliable also reliable under B: {(L & B).sum()} / {L.sum()}")
print("=" * 50)

# all 8 combinations (rows: right x left, columns: both)
pd.crosstab([R.rename("right"), L.rename("left")], B.rename("both"))

### Where to put the baseline window

Population mean around stimulus onset, for the cells reliable in any condition. The baseline for
the amplitude indices (Step 3) goes in a pre-stimulus stretch where nothing is evoked, so check
that the ball raise/lower doesn't drive a response there.

In [ ]:
def onset_aligned_mean(d, pre_s=4.0):
    """per-cell trial-averaged response aligned to on_start (cells x time), stimulus trials only"""
    t = d.frames_time                                   # ms
    dt = np.median(np.diff(t))
    npre = int(round(pre_s * 1000 / dt))
    acc, n = 0.0, 0
    for k in d.get_condition_trials():
        i0 = np.searchsorted(t, 1000 * d.on_start[k])
        if i0 - npre >= 0 and i0 + d.on_length <= len(t):
            acc = acc + d.response[:, i0 - npre:i0 + d.on_length]
            n += 1
    return acc / n, np.arange(-npre, d.on_length) * dt / 1000

resp_any = any_resp.to_numpy()
fig, ax = plt.subplots(figsize=(7, 3.5))
for eye in eyes:
    m, tt = onset_aligned_mean(D[eye])
    ax.plot(tt, m[resp_any].mean(axis=0), label=eye)
k = D["both"].get_condition_trials()
ax.axvline(0, color="k", lw=0.8, label="on_start")
ax.axvline(np.mean(D["both"].off_start[k] - D["both"].on_start[k]), color="gray", ls="--", label="off_start")
ax.set(xlabel="time from on_start (s)", ylabel="pooled z (cells reliable in any)",
       title="pick a baseline window where nothing is evoked")
ax.legend(frameon=False)
plt.show()